<a href="https://colab.research.google.com/github/SANGHATI23/neurofhir-qc/blob/main/13_NeuroFHIR_Review_WISH_Evidence_First_Study_Build.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# NeuroFHIR-Review — Notebook 13
## WISH 2026 Evidence-First Human–AI Study Build

This is the **next notebook after the frozen NeuroFHIR-QC / AMIA build**.

It creates a separate `wish_extension/` workspace and implements the shared WISH research layer without overwriting the existing reviewer application.

### What this notebook builds
- Evidence-First vs AI-First study protocol
- Path A / Path B reviewer logic
- 12 standardized/synthetic workflow scenarios
- wrong-but-plausible AI, uncertainty, QC failure, longitudinal discordance, and incomplete-provenance stress tests
- AI Evidence Passport for every scenario
- counterbalanced within-subject sequences
- reviewer training rubric
- interaction-log schema
- six-screen study application: Case brief → Evidence review → Initial judgment → AI reveal → AI Evidence Passport → Final action
- browser-side interaction logging and CSV/JSON export
- Notebook 13 readiness audit

**Scope boundary:** this is a formative human–AI interaction study build. The synthetic scenario manipulations do not create medical ground truth and do not constitute clinical validation.


In [1]:
# Cell 1 — Mount Drive, verify frozen NeuroFHIR-QC reviewer data, and create isolated WISH workspace

from __future__ import annotations

import csv
import hashlib
import json
import shutil
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

from google.colab import drive

drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/neurofhir-qc')
WISH_ROOT = PROJECT_ROOT / 'wish_extension'
CONFIG_ROOT = WISH_ROOT / 'config'
DATA_ROOT = WISH_ROOT / 'data'
SCHEMA_ROOT = WISH_ROOT / 'schemas'
FHIR_ROOT = WISH_ROOT / 'fhir'
DOC_ROOT = WISH_ROOT / 'docs'
EVAL_ROOT = WISH_ROOT / 'evaluation/results/notebook_13_review_foundation'
APP_ROOT = WISH_ROOT / 'app'
DEPLOY_ROOT = WISH_ROOT / 'submission/reviewer_application'

for folder in (CONFIG_ROOT, DATA_ROOT, SCHEMA_ROOT, FHIR_ROOT, DOC_ROOT, EVAL_ROOT, APP_ROOT, DEPLOY_ROOT):
    folder.mkdir(parents=True, exist_ok=True)

SOURCE_APP_DATA_CANDIDATES = [
    PROJECT_ROOT / 'app/frontend/public/data/app_data.json',
    PROJECT_ROOT / 'submission/reviewer_application/data/app_data.json',
]
SOURCE_APP_DATA_PATH = next((p for p in SOURCE_APP_DATA_CANDIDATES if p.exists() and p.stat().st_size > 0), None)
if SOURCE_APP_DATA_PATH is None:
    raise FileNotFoundError('Notebook 11 app_data.json not found. Run Notebook 11 successfully first.')

def utc_now() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace('+00:00','Z')

def load_json(path: Path) -> Any:
    with path.open('r', encoding='utf-8') as handle:
        return json.load(handle)

def write_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as handle:
        json.dump(payload, handle, indent=2, ensure_ascii=False, allow_nan=False)
        handle.write('\n')

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()

source_app_data = load_json(SOURCE_APP_DATA_PATH)
if len(source_app_data.get('cases', [])) != 3:
    raise AssertionError('Expected exactly 3 frozen NeuroFHIR-QC demonstration cases.')

SNAPSHOT_PATH = DATA_ROOT / 'neurofhir_qc_app_data_snapshot.json'
write_json(SNAPSHOT_PATH, source_app_data)
freeze_manifest = {
    'generated_utc': utc_now(),
    'source_app_data': str(SOURCE_APP_DATA_PATH),
    'source_app_data_sha256': sha256_file(SOURCE_APP_DATA_PATH),
    'source_case_count': len(source_app_data['cases']),
    'wish_workspace': str(WISH_ROOT),
    'original_neurofhir_qc_app_modified': False,
}
write_json(CONFIG_ROOT / 'source_freeze_manifest.json', freeze_manifest)

print('='*100)
print('✅ Frozen NeuroFHIR-QC reviewer data loaded')
print(f"✅ Source cases: {len(source_app_data['cases'])}")
print(f'✅ Isolated WISH workspace: {WISH_ROOT}')
print('✅ Original NeuroFHIR-QC app is not overwritten')
print('='*100)


Mounted at /content/drive
✅ Frozen NeuroFHIR-QC reviewer data loaded
✅ Source cases: 3
✅ Isolated WISH workspace: /content/drive/MyDrive/neurofhir-qc/wish_extension
✅ Original NeuroFHIR-QC app is not overwritten


In [2]:
# Cell 2 — Define the shared WISH research protocol

PROTOCOL_PATH = CONFIG_ROOT / 'research_protocol.json'
protocol = {
    'generated_utc': utc_now(),
    'project': 'NeuroFHIR-Review',
    'extension_of': 'NeuroFHIR-QC',
    'study_stage': 'formative/pilot human-AI interaction study',
    'shared_scientific_question': (
        'When AI-generated longitudinal neuroimaging evidence enters a review workflow, '
        'does requiring the human to inspect evidence and form an initial judgment BEFORE '
        'seeing the AI recommendation lead to better calibrated reliance, more appropriate '
        'escalation, and more accountable decision-making than showing AI first?'
    ),
    'conditions': {
        'evidence-first': {'ai_visible_before_initial_judgment': False},
        'ai-first': {'ai_visible_before_initial_judgment': True},
    },
    'path_A': {
        'reviewers': 'neuro specialist',
        'initial_labels': ['Stable','Progression','Uncertain'],
        'primary_outcomes': ['appropriate reliance','decision revision','error detection / escalation','confidence calibration','evidence/provenance inspection','qualitative reasoning'],
        'claim_boundary': 'Formative expert evaluation only; no patient-benefit, broad clinical-effectiveness, or deployment-safety claim.',
    },
    'path_B': {
        'reviewers': 'domain-adjacent or trained reviewer',
        'initial_labels': ['Accept','Flag','Escalate'],
        'primary_outcomes': ['correct workflow disposition','uncertainty sensitivity','provenance use','automation-bias proxy','confidence calibration','usability / reasoning themes'],
        'claim_boundary': 'Not clinical validation; no diagnostic-accuracy or clinician-performance claim.',
    },
    'final_actions': ['Accept AI','Keep initial judgment','Amend','Reject AI','Escalate'],
    'confidence_scale': [1,2,3,4,5],
    'reason_codes': [
        'evidence-supports-ai','ai-conflicts-with-longitudinal-evidence','low-ai-confidence','qc-failure',
        'missing-or-incomplete-provenance','model-limitation-relevant','evidence-incomplete',
        'uncertain-requires-expert-review','other'
    ],
    'screens': ['Case brief','Evidence review','Initial judgment','AI reveal','AI Evidence Passport','Final action'],
    'design': {'type':'within-subject crossover','scenario_count':12,'evidence_first_per_participant':6,'ai_first_per_participant':6,'counterbalance_sequences':['A','B']},
}
write_json(PROTOCOL_PATH, protocol)
print('✅ Shared Path A / Path B protocol written')


✅ Shared Path A / Path B protocol written


In [3]:
# Cell 3 — Create 12 standardized/synthetic study scenarios and counterbalancing

SCENARIO_JSON_PATH = DATA_ROOT / 'study_scenarios.json'
SCENARIO_CSV_PATH = DATA_ROOT / 'study_scenarios.csv'
COUNTERBALANCE_PATH = DATA_ROOT / 'counterbalance_sequences.json'

base_cases = {c['case_id']: c for c in source_app_data['cases']}
def resolve_case(preferred_id: str, fallback_index: int) -> dict[str, Any]:
    return deepcopy(base_cases.get(preferred_id, source_app_data['cases'][fallback_index]))

stable = resolve_case('stable', 0)
progression = resolve_case('progression', 1)
low_confidence = resolve_case('low-confidence', 2)

def make_scenario(sid, title, base, case_type, conclusion, conf_label, conf_score, qc_state, prov_state, longitudinal_relation, disposition, challenge, limitation):
    return {
        'scenario_id': sid,
        'title': title,
        'base_case_id': base['case_id'],
        'case_type': case_type,
        'synthetic_study_manipulation': True,
        'patient': deepcopy(base.get('patient', {})),
        'condition_context': deepcopy(base.get('condition', {})),
        'imaging_studies': deepcopy(base.get('imaging_studies', [])),
        'segmentation': deepcopy(base.get('segmentation', {})),
        'source_qc': deepcopy(base.get('qc', {})),
        'longitudinal': deepcopy(base.get('longitudinal', {})),
        'source_fhir': deepcopy(base.get('fhir', {})),
        'images': deepcopy(base.get('images', [])),
        'ai': {
            'conclusion': conclusion,
            'confidence_label': conf_label,
            'confidence_score': conf_score,
            'displayed_qc_state': qc_state,
            'provenance_state': prov_state,
            'longitudinal_concordance': longitudinal_relation,
            'known_limitation': limitation,
        },
        'reference_workflow_disposition_path_b': disposition,
        'expert_reference_required_for_path_a': True,
        'challenge': challenge,
    }

scenarios = [
    make_scenario('S01','Concordant stable / high confidence',stable,'concordant-high-confidence','Stable','high',0.91,'pass','complete','concordant','accept','Baseline benefit case.','AI confidence is a workflow signal, not a calibrated clinical probability.'),
    make_scenario('S02','Concordant progression / high confidence',progression,'concordant-high-confidence','Progression','high',0.93,'pass','complete','concordant','accept','Baseline progression case.','Interpretation depends on the displayed longitudinal evidence.'),
    make_scenario('S03','Correct but uncertain progression',progression,'correct-uncertain','Progression','low',0.42,'borderline','complete','concordant','flag','Tests response to uncertainty.','Low confidence should trigger additional evidence inspection.'),
    make_scenario('S04','Wrong but plausible: stable called progression',stable,'wrong-but-plausible','Progression','high',0.89,'pass','complete','discordant','escalate','Automation-bias stress test.','AI conclusion conflicts with the displayed longitudinal trajectory.'),
    make_scenario('S05','Wrong but plausible: progression called stable',progression,'wrong-but-plausible','Stable','high',0.90,'pass','complete','discordant','escalate','Opposite-direction automation-bias test.','AI conclusion conflicts with the displayed longitudinal trajectory.'),
    make_scenario('S06','Stable with low AI confidence',stable,'correct-uncertain','Stable','low',0.38,'borderline','complete','concordant','flag','Tests uncertainty sensitivity.','Borderline QC reduces confidence in autonomous workflow acceptance.'),
    make_scenario('S07','Progression with incomplete provenance',progression,'provenance-incomplete','Progression','high',0.88,'pass','incomplete','concordant','flag','Tests provenance use when AI appears correct.','Provenance is intentionally incomplete in this study stimulus.'),
    make_scenario('S08','Stable with incomplete provenance',stable,'provenance-incomplete','Stable','high',0.87,'pass','incomplete','concordant','flag','Tests provenance use without longitudinal conflict.','Provenance is intentionally incomplete in this study stimulus.'),
    make_scenario('S09','Persuasive AI conflicts with stable evidence',stable,'discordant-longitudinal-evidence','Progression','high',0.95,'pass','complete','discordant','escalate','Tests evidence use against persuasive AI.','High displayed confidence does not resolve contradictory longitudinal evidence.'),
    make_scenario('S10','Persuasive AI conflicts with progression evidence',progression,'discordant-longitudinal-evidence','Stable','high',0.94,'pass','complete','discordant','escalate','Second discordance stress test.','Displayed conclusion is intentionally discordant with source trajectory.'),
    make_scenario('S11','QC failure but confident AI',low_confidence,'qc-failure','Progression','high',0.92,'fail','complete','not-reliable','escalate','Tests whether QC failure blocks deference.','Source QC failed; longitudinal interpretation should not be treated as reliable.'),
    make_scenario('S12','QC failure with explicit AI uncertainty',low_confidence,'correct-uncertain','Uncertain','low',0.23,'fail','complete','not-reliable','escalate','Tests appropriate escalation when both AI and QC signal uncertainty.','Autonomous finalization should remain withheld.'),
]

for i, s in enumerate(scenarios, start=1):
    s['condition_by_sequence'] = {
        'A': 'evidence-first' if i % 2 == 1 else 'ai-first',
        'B': 'ai-first' if i % 2 == 1 else 'evidence-first',
    }

write_json(SCENARIO_JSON_PATH, {'generated_utc':utc_now(),'scenario_count':len(scenarios),'scenarios':scenarios})
write_json(COUNTERBALANCE_PATH, {'generated_utc':utc_now(),'A':{s['scenario_id']:s['condition_by_sequence']['A'] for s in scenarios},'B':{s['scenario_id']:s['condition_by_sequence']['B'] for s in scenarios}})

rows=[]
for s in scenarios:
    rows.append({
        'scenario_id':s['scenario_id'],'title':s['title'],'base_case_id':s['base_case_id'],'case_type':s['case_type'],
        'ai_conclusion':s['ai']['conclusion'],'ai_confidence_label':s['ai']['confidence_label'],'displayed_qc_state':s['ai']['displayed_qc_state'],
        'provenance_state':s['ai']['provenance_state'],'longitudinal_concordance':s['ai']['longitudinal_concordance'],
        'reference_workflow_disposition_path_b':s['reference_workflow_disposition_path_b'],'sequence_A':s['condition_by_sequence']['A'],'sequence_B':s['condition_by_sequence']['B']
    })
with SCENARIO_CSV_PATH.open('w', newline='', encoding='utf-8') as handle:
    writer=csv.DictWriter(handle, fieldnames=list(rows[0].keys())); writer.writeheader(); writer.writerows(rows)

for seq in ('A','B'):
    counts={'evidence-first':0,'ai-first':0}
    for s in scenarios: counts[s['condition_by_sequence'][seq]] += 1
    assert counts == {'evidence-first':6,'ai-first':6}, (seq, counts)

print('✅ 12 standardized/synthetic study scenarios generated')
print('✅ Both sequences contain 6 Evidence-First + 6 AI-First scenarios')


✅ 12 standardized/synthetic study scenarios generated
✅ Both sequences contain 6 Evidence-First + 6 AI-First scenarios


In [4]:
# Cell 4 — Create AI Evidence Passports, reviewer rubric, interaction schema, and FHIR mapping

PASSPORT_PATH = DATA_ROOT / 'ai_evidence_passports.json'
RUBRIC_PATH = DATA_ROOT / 'reviewer_rubric.json'
RUBRIC_MD_PATH = DOC_ROOT / 'REVIEWER_TRAINING_RUBRIC.md'
INTERACTION_SCHEMA_PATH = SCHEMA_ROOT / 'interaction_log_schema.json'
RESPONSE_TEMPLATE_PATH = SCHEMA_ROOT / 'reviewer_response_template.csv'
FHIR_MAPPING_PATH = FHIR_ROOT / 'study_writeback_mapping.json'

resources = source_app_data.get('representative_fhir_resources', {})
devices = [r for r in resources.values() if r.get('resourceType') == 'Device']
device = devices[0] if devices else {}
device_name = ((device.get('deviceName') or [{}])[0].get('name') or device.get('type',{}).get('text') or 'NeuroFHIR-QC segmentation model')

passports={}
for s in scenarios:
    complete = s['ai']['provenance_state'] == 'complete'
    passports[s['scenario_id']] = {
        'scenario_id':s['scenario_id'],
        'model':{'name':device_name,'device_reference':f"Device/{device.get('id')}" if device.get('id') else None},
        'intended_use':'Research workflow support for review of AI-derived longitudinal neuroimaging evidence; not diagnostic use.',
        'input_qc':{'displayed_state':s['ai']['displayed_qc_state'],'source_qc_category':s.get('source_qc',{}).get('category'),'source_qc_score':s.get('source_qc',{}).get('score')},
        'ai_confidence':{'label':s['ai']['confidence_label'],'display_score':s['ai']['confidence_score'],'warning':'Study/workflow signal; not a calibrated clinical probability.'},
        'longitudinal_concordance':s['ai']['longitudinal_concordance'],
        'provenance':{'state':s['ai']['provenance_state'],'complete':complete,'source_resources':s.get('source_fhir',{}).get('generated_resources',{}) if complete else {},'study_note':'Available for inspection.' if complete else 'Intentionally incomplete/hidden as a study manipulation.'},
        'known_limitation':s['ai']['known_limitation'],
    }
write_json(PASSPORT_PATH, {'generated_utc':utc_now(),'passport_count':len(passports),'passports':passports})

rubric = {
    'rules':[
        {'signal':'AI/evidence concordant + QC pass + provenance complete','path_B_action':'Accept'},
        {'signal':'Low AI confidence or borderline QC','path_B_action':'Flag'},
        {'signal':'AI conflicts with longitudinal evidence','path_B_action':'Escalate'},
        {'signal':'QC failure','path_B_action':'Escalate'},
        {'signal':'Missing/incomplete provenance','path_B_action':'Flag'},
        {'signal':'Evidence contradictory or insufficient','path_B_action':'Escalate'},
    ],
    'boundary':'Path B uses a workflow-disposition rubric, not a medical diagnostic rubric.'
}
write_json(RUBRIC_PATH, rubric)
RUBRIC_MD_PATH.write_text("# NeuroFHIR-Review — Reviewer Training Rubric\n\n| Signal | Path B action |\n|---|---|\n| AI/evidence concordant + QC pass + provenance complete | Accept |\n| Low AI confidence or borderline QC | Flag |\n| AI conflicts with longitudinal evidence | Escalate |\n| QC failure | Escalate |\n| Missing/incomplete provenance | Flag |\n| Evidence contradictory or insufficient | Escalate |\n\n**Boundary:** workflow-disposition rubric, not a medical diagnostic rubric.\n", encoding='utf-8')

fields=['session_id','participant_id','reviewer_tier','study_path','sequence','scenario_id','condition','event_type','event_utc','screen','initial_judgment','initial_confidence','ai_visible_before_initial_judgment','provenance_opened','final_action','final_confidence','reason_code','rationale','reference_workflow_disposition_path_b']
write_json(INTERACTION_SCHEMA_PATH, {'generated_utc':utc_now(),'schema_version':'1.0','fields':fields,'required_events':['case_opened','evidence_opened','initial_judgment_submitted','ai_exposed','passport_opened','provenance_opened','final_action_submitted']})
with RESPONSE_TEMPLATE_PATH.open('w', newline='', encoding='utf-8') as handle:
    csv.DictWriter(handle, fieldnames=fields).writeheader()

write_json(FHIR_MAPPING_PATH, {
    'AI-derived measurement':'Observation','AI interpretation':'DiagnosticReport','QC / uncertainty':'Observation or documented extension/profile approach',
    'Human review request':'Task','Model identity':'Device','Evidence lineage':'Provenance',
    'notebook_13_boundary':'Participant responses remain research-study data; Notebook 13 does not post participant responses to the public HAPI server.'
})

print(f'✅ AI Evidence Passports: {len(passports)}')
print('✅ Reviewer rubric + interaction schema + FHIR mapping created')


✅ AI Evidence Passports: 12
✅ Reviewer rubric + interaction schema + FHIR mapping created


In [5]:
# Cell 5 — Build a standalone six-screen NeuroFHIR-Review study application

APP_DATA_PATH = APP_ROOT / 'study_app_data.json'
APP_HTML_PATH = APP_ROOT / 'index.html'

study_app_data = {
    'generated_utc':utc_now(),
    'project':{'name':'NeuroFHIR-Review','extension_of':'NeuroFHIR-QC','environment':'WISH 2026 formative research prototype','repository_url':'https://github.com/SANGHATI23/neurofhir-qc','data_boundary':'Public de-identified research MRI + synthetic FHIR context + synthetic workflow manipulations.'},
    'protocol':protocol,
    'scenarios':scenarios,
    'passports':passports,
    'limitations':['Formative human-AI interaction prototype.','Synthetic workflow manipulations do not create medical ground truth.','Path B evaluates workflow behavior, not diagnostic accuracy.','No clinical deployment claim.'],
}
write_json(APP_DATA_PATH, study_app_data)

SOURCE_ASSET_ROOT = PROJECT_ROOT / 'app/frontend/public/assets'
APP_ASSET_ROOT = APP_ROOT / 'assets'
if APP_ASSET_ROOT.exists(): shutil.rmtree(APP_ASSET_ROOT)
APP_ASSET_ROOT.mkdir(parents=True, exist_ok=True)
asset_count=0
if SOURCE_ASSET_ROOT.exists():
    for src in SOURCE_ASSET_ROOT.rglob('*'):
        if not src.is_file(): continue
        rel=src.relative_to(SOURCE_ASSET_ROOT); dst=APP_ASSET_ROOT/rel; dst.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(src,dst); asset_count+=1

html = r"""<!doctype html>
<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>NeuroFHIR-Review | WISH Study</title>
<style>
:root{font-family:Inter,system-ui,-apple-system,Segoe UI,sans-serif;color:#15242d;background:#f4f7f9}*{box-sizing:border-box}body{margin:0}.shell{max-width:1080px;margin:auto;padding:28px 18px 60px}header{display:flex;justify-content:space-between;gap:20px;align-items:flex-start;margin-bottom:18px}h1{margin:5px 0;font-size:40px}h2,h3{margin-top:0}.eyebrow{text-transform:uppercase;letter-spacing:.08em;font-size:12px;font-weight:800;color:#0f766e}.card{background:#fff;border:1px solid #d9e3e8;border-radius:16px;padding:22px;margin:15px 0;box-shadow:0 8px 24px rgba(20,50,65,.06)}.grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(200px,1fr));gap:12px}.metric{padding:14px;border:1px solid #e0e8ec;border-radius:10px;background:#f8fafb}.metric span{display:block;color:#697b86;font-size:12px}.metric b{font-size:20px}.muted{color:#60717c}label{display:grid;gap:6px;margin:12px 0;font-weight:650}input,select,textarea,button{font:inherit}input,select,textarea{padding:10px;border:1px solid #bdcbd3;border-radius:9px;background:white}textarea{min-height:86px}button{padding:10px 14px;border:1px solid #b8c7cf;border-radius:10px;background:white;cursor:pointer}button.primary{background:#164e63;color:white;border-color:#164e63;font-weight:800}.notice{padding:12px 14px;background:#eef7f8;border-left:4px solid #0f766e;border-radius:8px;margin:14px 0}.warning{padding:10px 12px;background:#fff7ed;border-left:4px solid #c2410c;border-radius:8px}.steps{display:grid;grid-template-columns:repeat(6,1fr);gap:6px}.steps span{text-align:center;padding:8px;background:#e7edf1;border-radius:8px;font-size:12px}.steps .active{background:#164e63;color:white;font-weight:800}.steps .done{background:#dff4ef;color:#115e59}.ai{border:2px solid #3b82a0;background:#f2fbfd;border-radius:14px;padding:18px;margin:16px 0}.ai .big{font-size:30px;font-weight:850;margin:8px 0}.choices{display:grid;grid-template-columns:repeat(auto-fit,minmax(150px,1fr));gap:8px;margin:14px 0}.choices button.selected{background:#e8f5f7;border-color:#164e63;font-weight:800}.images{display:grid;grid-template-columns:repeat(3,1fr);gap:8px;margin:14px 0}.images img{width:100%;height:180px;object-fit:contain;background:#071017;border-radius:9px}pre{white-space:pre-wrap;overflow-wrap:anywhere;background:#0f1720;color:#d7edf2;padding:12px;border-radius:10px;max-height:300px;overflow:auto}.actions{display:flex;gap:10px;flex-wrap:wrap}.badge{padding:6px 10px;border-radius:999px;background:#dff4ef;color:#115e59;font-weight:800;font-size:12px}footer{text-align:center;color:#6a7b85;font-size:12px;margin-top:24px}@media(max-width:760px){header{display:block}.steps{grid-template-columns:repeat(3,1fr)}.images{grid-template-columns:1fr}}
</style></head><body><main class="shell"><header><div><div class="eyebrow">WISH 2026 formative study prototype</div><h1>NeuroFHIR-Review</h1><div class="muted">Evidence-first human–AI review for longitudinal neuroimaging evidence</div></div><div class="badge">FHIR R4 • Human-AI • Provenance</div></header><div id="app" class="card">Loading…</div><footer>Public de-identified MRI + synthetic FHIR context • Research prototype • No patient-care use</footer></main>
<script>
let D=null,state={started:false,participant:'',tier:'domain-adjacent',sequence:'A',order:[],index:0,screen:0,events:[],responses:{},initial:'',initialConf:3,finalAction:'',finalConf:3,reason:'',rationale:'',provenanceOpened:false,session:''};
const screens=['Case brief','Evidence review','Initial judgment','AI reveal','AI Evidence Passport','Final action']; const $=s=>document.querySelector(s); const now=()=>new Date().toISOString();
function esc(s){return String(s??'').replace(/[&<>\"']/g,m=>({'&':'&amp;','<':'&lt;','>':'&gt;','\"':'&quot;',"'":'&#039;'}[m]))}
function hash(s){let h=2166136261;for(let i=0;i<s.length;i++){h^=s.charCodeAt(i);h=Math.imul(h,16777619)}return h>>>0}
function shuffled(a,seed){let o=[...a],x=hash(seed);function r(){x=(Math.imul(x,1664525)+1013904223)>>>0;return x/4294967296}for(let i=o.length-1;i>0;i--){let j=Math.floor(r()*(i+1));[o[i],o[j]]=[o[j],o[i]]}return o}
function cur(){return state.order[state.index]} function condition(){return cur().condition_by_sequence[state.sequence]} function path(){return state.tier==='neuro-specialist'?'A':'B'}
function log(type,extra={}){if(!state.started||!cur())return;state.events.push({session_id:state.session,participant_id:state.participant,reviewer_tier:state.tier,study_path:path(),sequence:state.sequence,scenario_id:cur().scenario_id,condition:condition(),event_type:type,event_utc:now(),screen:screens[state.screen],reference_workflow_disposition_path_b:cur().reference_workflow_disposition_path_b,...extra});localStorage.setItem('neurofhir-review-events',JSON.stringify(state.events))}
function download(name,text,type='application/json'){let b=new Blob([text],{type}),u=URL.createObjectURL(b),a=document.createElement('a');a.href=u;a.download=name;a.click();URL.revokeObjectURL(u)}
function renderSteps(){return `<div class="steps">${screens.map((s,i)=>`<span class="${i===state.screen?'active':i<state.screen?'done':''}">${i+1}. ${esc(s)}</span>`).join('')}</div>`}
function startStudy(){let p=$('#participant').value.trim();if(!p){alert('Enter a pseudonymous participant ID.');return}state.participant=p;state.tier=$('#tier').value;state.sequence=$('#seq').value;state.order=shuffled(D.scenarios,p+'-'+state.sequence);state.started=true;state.session='NFR-'+p+'-'+Date.now();state.index=0;state.screen=0;state.events=[];state.responses={};resetFields();render();log('case_opened')}
function resetFields(){state.initial='';state.initialConf=3;state.finalAction='';state.finalConf=3;state.reason='';state.rationale='';state.provenanceOpened=false}
function setScreen(n){state.screen=n;render();log('screen_opened');if(n===1)log('evidence_opened');if(n===3)log('ai_exposed',{ai_visible_before_initial_judgment:condition()==='ai-first'});if(n===4)log('passport_opened')}
function chooseInitial(v){state.initial=v;render()} function chooseFinal(v){state.finalAction=v;render()}
function submitInitial(){if(!state.initial){alert('Choose an initial judgment.');return}state.initialConf=Number($('#initialConf').value);state.responses[cur().scenario_id]={...(state.responses[cur().scenario_id]||{}),scenario_id:cur().scenario_id,condition:condition(),initial_judgment:state.initial,initial_confidence:state.initialConf,initial_submitted_utc:now(),ai_conclusion:cur().ai.conclusion,ai_confidence_label:cur().ai.confidence_label,reference_workflow_disposition_path_b:cur().reference_workflow_disposition_path_b};log('initial_judgment_submitted',{initial_judgment:state.initial,initial_confidence:state.initialConf,ai_visible_before_initial_judgment:condition()==='ai-first'});setScreen(3)}
function toggleProv(){state.provenanceOpened=!state.provenanceOpened;if(state.provenanceOpened)log('provenance_opened',{provenance_opened:true});render()}
function submitFinal(){state.finalConf=Number($('#finalConf').value);state.reason=$('#reason').value;state.rationale=$('#rationale').value.slice(0,400);if(!state.finalAction||!state.reason){alert('Choose a final action and reason code.');return}state.responses[cur().scenario_id]={...(state.responses[cur().scenario_id]||{}),final_action:state.finalAction,final_confidence:state.finalConf,reason_code:state.reason,rationale:state.rationale,provenance_opened:state.provenanceOpened,final_submitted_utc:now()};log('final_action_submitted',{final_action:state.finalAction,final_confidence:state.finalConf,reason_code:state.reason,rationale:state.rationale,provenance_opened:state.provenanceOpened});state.index++;resetFields();state.screen=0;if(state.index<D.scenarios.length){render();log('case_opened')}else render()}
function exportJSON(){download(`neurofhir_review_${state.participant}_${state.sequence}.json`,JSON.stringify({exported_utc:now(),participant_id:state.participant,reviewer_tier:state.tier,study_path:path(),sequence:state.sequence,responses:state.responses,events:state.events},null,2))}
function csvEsc(v){return '\"'+String(v??'').replaceAll('\"','\"\"')+'\"'}
function exportCSV(){const cols=['participant_id','reviewer_tier','study_path','sequence','scenario_id','condition','initial_judgment','initial_confidence','ai_conclusion','ai_confidence_label','provenance_opened','final_action','final_confidence','reason_code','rationale','reference_workflow_disposition_path_b','initial_submitted_utc','final_submitted_utc'];const rows=Object.values(state.responses).map(r=>({participant_id:state.participant,reviewer_tier:state.tier,study_path:path(),sequence:state.sequence,...r}));const txt=[cols.map(csvEsc).join(','),...rows.map(r=>cols.map(c=>csvEsc(r[c])).join(','))].join('\n');download(`neurofhir_review_${state.participant}_${state.sequence}.csv`,txt,'text/csv')}
function AIBox(c){return `<div class="ai"><b>AI recommendation</b><div class="big">${esc(c.ai.conclusion)}</div><div class="grid"><div class="metric"><span>Confidence</span><b>${esc(c.ai.confidence_label)} (${c.ai.confidence_score})</b></div><div class="metric"><span>Displayed QC</span><b>${esc(c.ai.displayed_qc_state)}</b></div><div class="metric"><span>Longitudinal relation</span><b>${esc(c.ai.longitudinal_concordance)}</b></div></div><div class="warning">${esc(c.ai.known_limitation)}</div></div>`}
function render(){const app=$('#app');if(!state.started){app.innerHTML=`<h2>Study setup</h2><p class="muted">Use a pseudonymous participant ID. Do not enter PHI.</p><div class="grid"><label>Participant ID<input id="participant" placeholder="P001"></label><label>Reviewer tier<select id="tier"><option value="neuro-specialist">Neuro specialist — Path A</option><option value="domain-adjacent" selected>Clinical/biomedical/imaging-AI expert — Path B1</option><option value="trained-reviewer">Trained reviewer — Path B2</option></select></label><label>Counterbalance sequence<select id="seq"><option>A</option><option>B</option></select></label></div><div class="notice"><b>Shared question:</b> ${esc(D.protocol.shared_scientific_question)}</div><button class="primary" onclick="startStudy()">Start 12-case pilot</button>`;return}
if(state.index>=D.scenarios.length){app.innerHTML=`<h2>Session complete</h2><p>Export both files. Notebook 14 will ingest these pilot exports.</p><div class="actions"><button class="primary" onclick="exportCSV()">Export case-level CSV</button><button onclick="exportJSON()">Export full event JSON</button></div>`;return}
const c=cur(),cond=condition(),aiEarly=cond==='ai-first',p=path(),pass=D.passports[c.scenario_id];let body=`<div style="display:flex;justify-content:space-between;gap:12px"><div><div class="eyebrow">Path ${p}</div><h2>${esc(c.scenario_id)} — ${esc(c.title)}</h2></div><div class="badge">${cond==='evidence-first'?'Evidence-First':'AI-First'} • ${state.index+1}/12</div></div>${renderSteps()}`;
if(state.screen===0)body+=`<h3>1. Case brief</h3><p class="muted">No AI recommendation is shown on this screen.</p><div class="grid"><div class="metric"><span>Scenario</span><b>${esc(c.scenario_id)}</b></div><div class="metric"><span>Base evidence case</span><b>${esc(c.base_case_id)}</b></div><div class="metric"><span>Context</span><b>Synthetic FHIR + public MRI</b></div></div><p><button class="primary" onclick="setScreen(1)">Review evidence</button></p>`;
if(state.screen===1){let pct=c.longitudinal?.percent_change,imgs=(c.images||[]).slice(0,3).map(x=>`<img src="${esc(x.replace('./assets/','assets/'))}" alt="evidence">`).join('');body+=`<h3>2. Evidence review</h3><div class="grid"><div class="metric"><span>Prior volume</span><b>${esc(c.longitudinal?.prior_volume_ml??'—')} mL</b></div><div class="metric"><span>Current volume</span><b>${esc(c.longitudinal?.current_volume_ml??'—')} mL</b></div><div class="metric"><span>Longitudinal change</span><b>${pct==null?'—':Number(pct).toFixed(2)+'%'}</b></div><div class="metric"><span>Source QC</span><b>${esc(c.source_qc?.category??'—')}</b></div></div>${imgs?`<div class="images">${imgs}</div>`:''}<div class="notice"><b>Study QC state:</b> ${esc(c.ai.displayed_qc_state)} • <b>Provenance:</b> ${esc(c.ai.provenance_state)}</div>${aiEarly?AIBox(c):''}<button class="primary" onclick="setScreen(2)">Record first judgment</button>`}
if(state.screen===2){let opts=p==='A'?D.protocol.path_A.initial_labels:D.protocol.path_B.initial_labels;body+=`<h3>3. Initial judgment</h3><p class="muted">${aiEarly?'AI advice has already been visible in this AI-First condition.':'AI advice remains hidden until you submit this judgment.'}</p><div class="choices">${opts.map(o=>`<button class="${state.initial===o?'selected':''}" onclick='chooseInitial(${JSON.stringify(o)})'>${esc(o)}</button>`).join('')}</div><label>Confidence: <b>${state.initialConf}/5</b><input id="initialConf" type="range" min="1" max="5" value="${state.initialConf}" oninput="state.initialConf=Number(this.value);render()"></label><button class="primary" onclick="submitInitial()">Submit judgment</button>`}
if(state.screen===3)body+=`<h3>4. AI reveal</h3><p class="muted">${aiEarly?'The recommendation was visible before your first judgment.':'The recommendation is now revealed after your independent judgment.'}</p>${AIBox(c)}<button class="primary" onclick="setScreen(4)">Inspect AI Evidence Passport</button>`;
if(state.screen===4)body+=`<h3>5. AI Evidence Passport</h3><div class="grid"><div class="metric"><span>Model</span><b>${esc(pass.model.name)}</b></div><div class="metric"><span>Intended use</span><b>${esc(pass.intended_use)}</b></div><div class="metric"><span>Input QC</span><b>${esc(pass.input_qc.displayed_state)}</b></div><div class="metric"><span>Provenance</span><b>${esc(pass.provenance.state)}</b></div></div><div class="warning">${esc(pass.known_limitation)}</div><p><button onclick="toggleProv()">${state.provenanceOpened?'Hide provenance chain':'Open provenance chain'}</button></p>${state.provenanceOpened?`<pre>${esc(JSON.stringify(pass.provenance,null,2))}</pre>`:''}<button class="primary" onclick="setScreen(5)">Make final decision</button>`;
if(state.screen===5)body+=`<h3>6. Final action</h3><div class="choices">${D.protocol.final_actions.map(o=>`<button class="${state.finalAction===o?'selected':''}" onclick='chooseFinal(${JSON.stringify(o)})'>${esc(o)}</button>`).join('')}</div><label>Final confidence: <b>${state.finalConf}/5</b><input id="finalConf" type="range" min="1" max="5" value="${state.finalConf}" oninput="state.finalConf=Number(this.value);render()"></label><label>Reason code<select id="reason"><option value="">Choose…</option>${D.protocol.reason_codes.map(r=>`<option value="${esc(r)}" ${state.reason===r?'selected':''}>${esc(r)}</option>`).join('')}</select></label><label>Short rationale (optional)<textarea id="rationale" maxlength="400">${esc(state.rationale)}</textarea></label><div class="notice"><b>Study boundary:</b> ${p==='A'?'Formative expert reliance/escalation study; not clinical deployment validation.':'Workflow disposition task; not medical diagnostic accuracy.'}</div><button class="primary" onclick="submitFinal()">Submit and continue</button>`;
app.innerHTML=body}
fetch('study_app_data.json').then(r=>r.json()).then(x=>{D=x;render()}).catch(e=>{$('#app').innerHTML='<b>Failed to load study data:</b> '+esc(e)})
</script></body></html>"""

APP_HTML_PATH.write_text(html, encoding='utf-8')
if DEPLOY_ROOT.exists(): shutil.rmtree(DEPLOY_ROOT)
shutil.copytree(APP_ROOT, DEPLOY_ROOT)
assert (DEPLOY_ROOT/'index.html').exists()
assert (DEPLOY_ROOT/'study_app_data.json').exists()
print('✅ Six-screen NeuroFHIR-Review study app generated')
print(f'✅ Source app: {APP_ROOT}')
print(f'✅ Deployable app: {DEPLOY_ROOT}')
print(f'✅ Copied visual assets: {asset_count}')
print('✅ Browser interaction logging + CSV/JSON export included')


✅ Six-screen NeuroFHIR-Review study app generated
✅ Source app: /content/drive/MyDrive/neurofhir-qc/wish_extension/app
✅ Deployable app: /content/drive/MyDrive/neurofhir-qc/wish_extension/submission/reviewer_application
✅ Copied visual assets: 9
✅ Browser interaction logging + CSV/JSON export included


In [6]:
# Cell 6 — Final audit and handoff to Notebook 14

AUDIT_PATH = EVAL_ROOT / 'notebook_13_neurofhir_review_wish_build_audit.json'
AUDIT_MD_PATH = DOC_ROOT / 'NOTEBOOK_13_NEUROFHIR_REVIEW_WISH_BUILD.md'

sequence_counts={}
for seq in ('A','B'):
    counts={'evidence-first':0,'ai-first':0}
    for s in scenarios: counts[s['condition_by_sequence'][seq]] += 1
    sequence_counts[seq]=counts

case_type_counts={}
for s in scenarios: case_type_counts[s['case_type']] = case_type_counts.get(s['case_type'],0)+1

required=[PROTOCOL_PATH,SCENARIO_JSON_PATH,SCENARIO_CSV_PATH,COUNTERBALANCE_PATH,PASSPORT_PATH,RUBRIC_PATH,RUBRIC_MD_PATH,INTERACTION_SCHEMA_PATH,RESPONSE_TEMPLATE_PATH,FHIR_MAPPING_PATH,APP_DATA_PATH,APP_HTML_PATH,DEPLOY_ROOT/'index.html',DEPLOY_ROOT/'study_app_data.json']
missing=[str(p) for p in required if not p.exists() or p.stat().st_size==0]
if missing: raise AssertionError('Notebook 13 final gate failed. Missing:\n'+'\n'.join(' - '+p for p in missing))

final_gate = len(scenarios)==12 and len(passports)==12 and sequence_counts['A']=={'evidence-first':6,'ai-first':6} and sequence_counts['B']=={'evidence-first':6,'ai-first':6}
audit={
    'status':'completed' if final_gate else 'failed','audited_utc':utc_now(),'notebook':'13_NeuroFHIR_Review_WISH_Evidence_First_Study_Build.ipynb','source_neurofhir_qc_frozen':True,
    'metrics':{'base_case_count':3,'study_scenario_count':len(scenarios),'passport_count':len(passports),'screen_count':6,'sequence_counts':sequence_counts,'case_type_counts':case_type_counts,'deployable_study_app_created':True},
    'implemented':['isolated WISH extension','Path A / Path B protocol','Evidence-First vs AI-First manipulation','12 standardized/synthetic scenarios','AI Evidence Passport','initial judgment + confidence capture','final reconciliation + confidence + reason capture','provenance inspection logging','CSV/JSON export','reviewer rubric','FHIR representation mapping','deployable six-screen study app'],
    'next_notebook':'Notebook 14 — Pilot Data Ingestion and WISH Human-AI Analysis',
    'next_analysis':['ingest participant CSV/JSON exports','compare Evidence-First vs AI-First','measure escalation and provenance inspection','measure confidence change','analyze wrong-but-plausible AI scenarios separately','calculate Path B workflow-disposition accuracy','calculate Path A appropriate reliance if expert reference is available','generate WISH figures/tables'],
    'final_gate':final_gate,
}
write_json(AUDIT_PATH,audit)
AUDIT_MD_PATH.write_text('# Notebook 13 — NeuroFHIR-Review WISH Build\n\n**Status:** '+audit['status']+'\n\n## Completed\n- Separate wish_extension workspace.\n- 12 standardized/synthetic scenarios.\n- 6 Evidence-First + 6 AI-First cases per sequence.\n- Six-screen human-AI review workflow.\n- AI Evidence Passport.\n- Interaction logging and participant export.\n- Path A / Path B claim boundaries.\n\n## Next\nNotebook 14 should ingest actual pilot exports and run the WISH analysis.\n', encoding='utf-8')

print('='*100)
print(f'✅ NOTEBOOK 13 FINAL GATE: {final_gate}')
print(f'✅ Scenarios: {len(scenarios)}')
print(f'✅ Passports: {len(passports)}')
print(f"✅ Sequence A: {sequence_counts['A']}")
print(f"✅ Sequence B: {sequence_counts['B']}")
print(f'✅ Deployable app: {DEPLOY_ROOT}')
print(f'✅ Audit: {AUDIT_PATH}')
print('NEXT: Notebook 14 — ingest real pilot exports and perform WISH analysis.')
print('='*100)


✅ NOTEBOOK 13 FINAL GATE: True
✅ Scenarios: 12
✅ Passports: 12
✅ Sequence A: {'evidence-first': 6, 'ai-first': 6}
✅ Sequence B: {'evidence-first': 6, 'ai-first': 6}
✅ Deployable app: /content/drive/MyDrive/neurofhir-qc/wish_extension/submission/reviewer_application
✅ Audit: /content/drive/MyDrive/neurofhir-qc/wish_extension/evaluation/results/notebook_13_review_foundation/notebook_13_neurofhir_review_wish_build_audit.json
NEXT: Notebook 14 — ingest real pilot exports and perform WISH analysis.
